# Build cost table

Produces `data/cost_model/cost_table.csv`. One row per (symbol, venue, week_start) with `half_spread_bps`, `impact_bps_per_unit`, `taker_fee_bps`, and a `half_spread_source` column tagging where the half spread came from.

## Output

One row per (symbol, venue, week_start). Columns.

- `symbol`. Ticker string.
- `venue`. 0 futures, 1 spot (matches the C++ Portfolio layout).
- `week_start`. ISO Monday of the week, `YYYY-MM-DD`.
- `half_spread_bps`. Half spread in bps.
- `half_spread_source`. Provenance of the half spread. `book` measured from futures bookTicker, `book_median` per-symbol book fill outside the bookTicker window, `futures_book` / `futures_book_median` the futures value carried onto the spot row.
- `impact_bps_per_unit`. Weekly median Amihud illiquidity from aggTrades, bps per unit qty.
- `taker_fee_bps`. 4.0 futures VIP0, 10.0 spot VIP0.
- `n_days_book`, `n_days_trade`, `n_bars`. Sample counts contributing to each column.

Venues follow the engine, 0 futures and 1 spot. The futures half spread is measured from bookTicker where the archive has it and filled with a per-symbol book median elsewhere. Binance publishes no spot bookTicker, and no kline or trade estimator resolves the spot spread on the liquid majors, so the spot half spread is the futures value carried onto the spot row. Spot impact and taker fee are measured on their own data.

In [1]:
import io
import time
import zipfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/qp-cost-table")
    KLINES_ROOT = None
except ImportError:
    # Walk up until a directory containing "data/binance_historical" is found.
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "data" / "binance_historical").is_dir():
        ROOT = ROOT.parent
    KLINES_ROOT = ROOT / "data" / "binance_historical"
    ROOT = ROOT / "data" / "cost_model"

Mounted at /content/drive


Symbols must match `include/data_source/source/venue/binance/binance_historical/bin_hist_symbol_table.hpp`'s `detail::kSymbols`.

In [2]:
SYMBOLS   = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
             "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]
FIRST_DAY = date(2022, 1, 1)
LAST_DAY  = date(2024, 12, 31)

# Binance USD-M VIP0 taker fee.
TAKER_FEE_BPS = 4.0
FUTURES_VENUE = 0  # engine convention, futures leg is venue 0, spot is 1

# Spot leg. 10 bps taker, distinct from futures. Impact from spot aggTrades.
# No spot bookTicker exists, so the half spread is proxied from the futures
# book row.
SPOT_VENUE = 1
SPOT_TAKER_FEE_BPS = 10.0

BOOK_URL       = "https://data.binance.vision/data/futures/um/daily/bookTicker/{s}/{s}-bookTicker-{d}.zip"
TRADE_URL      = "https://data.binance.vision/data/futures/um/daily/aggTrades/{s}/{s}-aggTrades-{d}.zip"
SPOT_TRADE_URL = "https://data.binance.vision/data/spot/daily/aggTrades/{s}/{s}-aggTrades-{d}.zip"

ROOT.mkdir(parents=True, exist_ok=True)

BOOK_CHECKPOINT       = ROOT / "book_daily.csv"
TRADE_CHECKPOINT      = ROOT / "trade_daily.csv"
SPOT_TRADE_CHECKPOINT = ROOT / "spot_trade_daily.csv"
INTERMEDIATE_PATH     = ROOT / "book_trade_weekly.csv"
FINAL_PATH            = ROOT / "cost_table.csv"

Streams each daily zip in memory, extracts, summarizes, discards. Raw never touches disk.

In [3]:
REQUEST_TIMEOUT = 60
RETRY_SLEEP     = 5

# aggTrades dumps have no header row for most months, some newer months do.
# Spot carries an extra is_best_match column the futures dumps lack.
AGG_TRADES_COLS      = ["agg_trade_id", "price", "quantity", "first_trade_id",
                        "last_trade_id", "transact_time", "is_buyer_maker"]
SPOT_AGG_TRADES_COLS = AGG_TRADES_COLS + ["is_best_match"]

def _download_zip(url):
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.content
        except requests.RequestException as e:
            if attempt == 2:
                raise
            print(f"  retry {attempt + 1} on {url}: {e}")
            time.sleep(RETRY_SLEEP)

def _read_zipped_csv(zip_bytes, **read_csv_kwargs):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        name = next((n for n in z.namelist() if n.endswith(".csv")), None)
        if name is None:
            return None
        with z.open(name) as f:
            return pd.read_csv(f, **read_csv_kwargs)

def summarize_book_ticker(symbol, day):
    blob = _download_zip(BOOK_URL.format(s=symbol, d=day.isoformat()))
    if blob is None:
        return None
    # Median over ~2k evenly-spaced rows is indistinguishable from
    # the median over ~10M
    df = _read_zipped_csv(blob, skiprows=lambda i: i > 0 and i % 500 != 0)
    if df is None or df.empty:
        return None
    bid = df["best_bid_price"].astype(float)
    ask = df["best_ask_price"].astype(float)
    ok  = (bid > 0) & (ask > 0) & (ask >= bid)
    half_spread_bps = 1e4 * ((ask[ok] - bid[ok]) / (ask[ok] + bid[ok]))
    return {
        "symbol":          symbol,
        "day":             day.isoformat(),
        "half_spread_bps": float(half_spread_bps.median()),
        "n_events":        int(ok.sum()),
    }

def summarize_agg_trades(symbol, day, url_tmpl=TRADE_URL, names=AGG_TRADES_COLS):
    blob = _download_zip(url_tmpl.format(s=symbol, d=day.isoformat()))
    if blob is None:
        return None
    df = _read_zipped_csv(blob, header=None, names=names, low_memory=False)
    if df is None or df.empty:
        return None
    # Drop a header row when a month happens to have one.
    try:
        float(df.iloc[0]["price"])
    except (ValueError, TypeError):
        df = df.iloc[1:]
    if df.empty:
        return None
    price = pd.to_numeric(df["price"], errors="coerce").to_numpy()
    qty   = pd.to_numeric(df["quantity"], errors="coerce").to_numpy()
    ts    = pd.to_numeric(df["transact_time"], errors="coerce").to_numpy()
    order = np.argsort(ts)
    price = price[order]
    qty   = qty[order]
    if len(price) < 2:
        return None
    with np.errstate(divide="ignore", invalid="ignore"):
        log_ret  = np.abs(np.diff(np.log(price)))
        per_unit = log_ret / np.where(qty[:-1] > 0, qty[:-1], np.nan)
    per_unit = per_unit[np.isfinite(per_unit)]
    if per_unit.size == 0:
        return None
    return {
        "symbol":              symbol,
        "day":                 day.isoformat(),
        "impact_bps_per_unit": float(1e4 * np.median(per_unit)),
        "n_trades":            int(len(price)),
    }

Iterates `(symbol, day, kind)`, resumes from checkpoints, flushes per symbol.

In [4]:
def _load_or_empty(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

def _done_pairs(df):
    if df.empty:
        return set()
    return set(zip(df["symbol"], df["day"]))

def _days_in_range(first, last):
    d = first
    while d <= last:
        yield d
        d = d + timedelta(days=1)

def run(kinds=("book", "trade", "spot_trade"), symbols=None):
    syms = symbols if symbols is not None else SYMBOLS

    book_rows       = _load_or_empty(BOOK_CHECKPOINT).to_dict("records")
    trade_rows      = _load_or_empty(TRADE_CHECKPOINT).to_dict("records")
    spot_trade_rows = _load_or_empty(SPOT_TRADE_CHECKPOINT).to_dict("records")
    book_done       = _done_pairs(_load_or_empty(BOOK_CHECKPOINT))
    trade_done      = _done_pairs(_load_or_empty(TRADE_CHECKPOINT))
    spot_trade_done = _done_pairs(_load_or_empty(SPOT_TRADE_CHECKPOINT))

    for symbol in syms:
        print(f"=== {symbol} ===")
        for day in _days_in_range(FIRST_DAY, LAST_DAY):
            key = (symbol, day.isoformat())
            if "book" in kinds and key not in book_done:
                row = summarize_book_ticker(symbol, day)
                if row is not None:
                    book_rows.append(row)
            if "trade" in kinds and key not in trade_done:
                row = summarize_agg_trades(symbol, day)
                if row is not None:
                    trade_rows.append(row)
            if "spot_trade" in kinds and key not in spot_trade_done:
                row = summarize_agg_trades(symbol, day, url_tmpl=SPOT_TRADE_URL,
                                           names=SPOT_AGG_TRADES_COLS)
                if row is not None:
                    spot_trade_rows.append(row)
        if "book" in kinds:
            pd.DataFrame(book_rows).to_csv(BOOK_CHECKPOINT, index=False)
        if "trade" in kinds:
            pd.DataFrame(trade_rows).to_csv(TRADE_CHECKPOINT, index=False)
        if "spot_trade" in kinds:
            pd.DataFrame(spot_trade_rows).to_csv(SPOT_TRADE_CHECKPOINT, index=False)

# Full build fetches all three, resuming from any existing checkpoints.
run(kinds=("book", "trade", "spot_trade"))

=== BTCUSDT ===
=== ETHUSDT ===
=== SOLUSDT ===
=== BNBUSDT ===
=== XRPUSDT ===
=== DOGEUSDT ===
=== ADAUSDT ===
=== LINKUSDT ===
=== AVAXUSDT ===
=== LTCUSDT ===


Reads the per-day checkpoints, buckets by ISO Monday, medians per (symbol, week). Futures rows carry the book half spread and futures impact, spot rows carry spot impact. Writes `book_trade_weekly.csv`.

In [5]:
def _week_start(day_str):
    d = date.fromisoformat(day_str)
    return (d - timedelta(days=d.weekday())).isoformat()

def _weekly_impact(trade, venue, fee):
    cols = ["symbol", "venue", "week_start", "impact_bps_per_unit",
            "n_days_trade", "taker_fee_bps"]
    if trade.empty:
        return pd.DataFrame(columns=cols)
    trade = trade.copy()
    trade["week_start"] = trade["day"].map(_week_start)
    w = trade.groupby(["symbol", "week_start"], as_index=False).agg(
        impact_bps_per_unit=("impact_bps_per_unit", "median"),
        n_days_trade=("day", "count"),
    )
    w["venue"]         = venue
    w["taker_fee_bps"] = fee
    return w[cols]

def build_intermediate_table():
    book       = _load_or_empty(BOOK_CHECKPOINT)
    trade      = _load_or_empty(TRADE_CHECKPOINT)
    spot_trade = _load_or_empty(SPOT_TRADE_CHECKPOINT)
    if book.empty and trade.empty and spot_trade.empty:
        raise RuntimeError("no checkpoints yet, run() first")

    if not book.empty:
        book = book.copy()
        book["week_start"] = book["day"].map(_week_start)
        book_weekly = book.groupby(["symbol", "week_start"], as_index=False).agg(
            half_spread_bps_book=("half_spread_bps", "median"),
            n_days_book=("day", "count"),
        )
    else:
        book_weekly = pd.DataFrame(columns=["symbol", "week_start",
                                            "half_spread_bps_book", "n_days_book"])

    fut_impact = _weekly_impact(trade, FUTURES_VENUE, TAKER_FEE_BPS)
    futures = book_weekly.merge(
        fut_impact.drop(columns=["venue", "taker_fee_bps"]),
        on=["symbol", "week_start"], how="outer")
    futures["venue"]         = FUTURES_VENUE
    futures["taker_fee_bps"] = TAKER_FEE_BPS

    spot = _weekly_impact(spot_trade, SPOT_VENUE, SPOT_TAKER_FEE_BPS)
    spot["half_spread_bps_book"] = np.nan
    spot["n_days_book"]          = np.nan

    cols = ["symbol", "venue", "week_start", "half_spread_bps_book",
            "impact_bps_per_unit", "taker_fee_bps", "n_days_book", "n_days_trade"]
    weekly = pd.concat([futures[cols], spot[cols]], ignore_index=True)
    weekly = weekly.sort_values(["symbol", "venue", "week_start"]).reset_index(drop=True)

    weekly.to_csv(INTERMEDIATE_PATH, index=False)
    print(f"wrote {INTERMEDIATE_PATH}  rows={len(weekly)}")
    return weekly

build_intermediate_table()

wrote /content/drive/MyDrive/qp-cost-table/book_trade_weekly.csv  rows=1580


/tmp/ipykernel_5747/3738191757.py:51: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weekly = pd.concat([futures[cols], spot[cols]], ignore_index=True)


,symbol,venue,week_start,half_spread_bps_book,impact_bps_per_unit,taker_fee_bps,n_days_book,n_days_trade
0,ADAUSDT,1,2021-12-27,NaN,0.000000,10.0,NaN,2
1,ADAUSDT,1,2022-01-03,NaN,0.000000,10.0,NaN,7
2,ADAUSDT,1,2022-01-10,NaN,0.000000,10.0,NaN,7
3,ADAUSDT,1,2022-01-17,NaN,0.000000,10.0,NaN,7
4,ADAUSDT,1,2022-01-24,NaN,0.000000,10.0,NaN,7
...,...,...,...,...,...,...,...,...
1575,XRPUSDT,1,2024-12-02,NaN,0.000234,10.0,NaN,7
1576,XRPUSDT,1,2024-12-09,NaN,0.000000,10.0,NaN,7
1577,XRPUSDT,1,2024-12-16,NaN,0.000777,10.0,NaN,7
1578,XRPUSDT,1,2024-12-23,NaN,0.000468,10.0,NaN,7


## Abdi-Ranaldo half-spread from klines

Reads local 1 min futures klines, computes Abdi-Ranaldo half spread per (symbol, ISO-week).

$s^2 = 4 E[(c_t - eta_t)(c_t - eta_{t+1})]$ where $eta_t = ln((H_t + L_t) / 2)$ and $c_t = ln(close_t)$.

Half spread is $s / 2 = sqrt(E[...])$.

In [6]:
KLINE_COLS = ["symbol", "kind", "open_time", "open", "high", "low", "close",
              "volume", "close_time", "quote_volume", "count",
              "taker_buy_base", "taker_buy_quote", "ignore"]

def _load_klines(symbol):
    if KLINES_ROOT is None:
        raise RuntimeError("Klines not found")
    d = KLINES_ROOT / symbol / "futures" / "klines"
    files = sorted(d.glob(f"{symbol}-1m-*.csv"))
    if not files:
        raise FileNotFoundError(f"no klines under {d}")
    parts = []
    for f in files:
        parts.append(pd.read_csv(f, header=None, names=KLINE_COLS,
                                 usecols=["open_time", "high", "low", "close"]))
    df = pd.concat(parts, ignore_index=True)
    df["ts"]    = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["high"]  = df["high"].astype(float)
    df["low"]   = df["low"].astype(float)
    df["close"] = df["close"].astype(float)
    df = df.sort_values("ts").drop_duplicates("ts").reset_index(drop=True)
    return df[["ts", "high", "low", "close"]]

def _week_start_from_ts(ts):
    d = ts.date()
    return (d - timedelta(days=d.weekday())).isoformat()

def _ar_by_week(klines):
    k   = klines.loc[(klines["high"] > 0) & (klines["low"] > 0)
                     & (klines["close"] > 0)].reset_index(drop=True)
    eta = np.log((k["high"].to_numpy() + k["low"].to_numpy()) / 2.0)
    c   = np.log(k["close"].to_numpy())
    term = (c[:-1] - eta[:-1]) * (c[:-1] - eta[1:])
    bucket = k["ts"].iloc[:-1].map(_week_start_from_ts).to_numpy()
    df = pd.DataFrame({"week_start": bucket, "term": term})
    weekly = df.groupby("week_start", as_index=False).agg(
        mean_term=("term", "mean"),
        n_bars=("term", "count"),
    )
    weekly["half_spread_bps_ar"] = 1e4 * np.sqrt(np.clip(weekly["mean_term"], 0.0, None))
    return weekly[["week_start", "half_spread_bps_ar", "n_bars"]]

def build_ar_table(symbols=None):
    rows = []
    for symbol in (symbols or SYMBOLS):
        print(f"  ar: {symbol}")
        w = _ar_by_week(_load_klines(symbol))
        w["symbol"] = symbol
        w["venue"]  = FUTURES_VENUE
        rows.append(w)
    return pd.concat(rows, ignore_index=True)

ar_table = build_ar_table()
ar_table.head()

  ar: BTCUSDT


RuntimeError: Klines not found

# Calibration and merge

Reads `book_trade_weekly.csv`, joins with AR, prints per-symbol book-vs-AR agreement over the bookTicker overlap window. Fills the futures half spread, carries it onto the spot rows, writes `cost_table.csv`.

In [ ]:
def _calibrate(merged):
    overlap = merged.dropna(subset=["half_spread_bps_book", "half_spread_bps_ar"])
    if overlap.empty:
        print("no overlap between book and AR windows; skipping calibration")
        return
    print()
    print("=== calibration (book vs AR over overlap weeks) ===")
    print(f"{'symbol':10s} {'weeks':>6s} {'corr':>6s} {'book_med':>10s} {'ar_med':>10s} {'ratio':>7s}")
    for symbol, g in overlap.groupby("symbol"):
        b = g["half_spread_bps_book"].to_numpy()
        a = g["half_spread_bps_ar"].to_numpy()
        corr  = float(np.corrcoef(b, a)[0, 1]) if len(g) > 2 else float("nan")
        b_med = float(np.median(b))
        a_med = float(np.median(a))
        ratio = a_med / b_med if b_med > 0 else float("nan")
        print(f"{symbol:10s} {len(g):>6d} {corr:>6.2f} {b_med:>10.3f} {a_med:>10.3f} {ratio:>7.2f}")
    print("=====================================================")


def finalize():
    if not INTERMEDIATE_PATH.exists():
        raise FileNotFoundError(
            f"expected {INTERMEDIATE_PATH}. Move the Colab output there first.")

    inter = pd.read_csv(INTERMEDIATE_PATH)
    ar    = build_ar_table()

    merged = inter.merge(ar, on=["symbol", "venue", "week_start"], how="outer")

    # AR is reported only, book is the futures source of record.
    _calibrate(merged)

    # Futures half spread. Measured book where present, per-symbol book median
    # in the months outside the bookTicker archive.
    book        = merged["half_spread_bps_book"]
    book_median = merged.groupby("symbol")["half_spread_bps_book"].transform("median")
    merged["half_spread_bps"]    = book.where(book.notna(), book_median)
    merged["half_spread_source"] = np.where(book.notna(), "book", "book_median")
    merged["taker_fee_bps"]      = merged["taker_fee_bps"].fillna(TAKER_FEE_BPS)
    merged["venue"]              = merged["venue"].astype(int)

    # Spot has no bookTicker and no estimator resolves it. Carry the futures
    # half spread onto the spot row, keyed by week, provenance prefixed futures.
    fut  = merged[merged["venue"] == FUTURES_VENUE].set_index(["symbol", "week_start"])
    mask = merged["venue"] == SPOT_VENUE
    key  = list(zip(merged.loc[mask, "symbol"], merged.loc[mask, "week_start"]))
    merged.loc[mask, "half_spread_bps"] = fut["half_spread_bps"].reindex(key).to_numpy()
    src = fut["half_spread_source"].reindex(key).to_numpy()
    merged.loc[mask, "half_spread_source"] = [
        "futures_" + s if isinstance(s, str) else s for s in src]

    cols = ["symbol", "venue", "week_start", "half_spread_bps",
            "half_spread_source", "impact_bps_per_unit",
            "taker_fee_bps", "n_days_book", "n_days_trade", "n_bars"]
    final = merged[cols].copy()
    final = final.sort_values(["symbol", "venue", "week_start"]).reset_index(drop=True)

    assert set(final["venue"].unique()) == {FUTURES_VENUE, SPOT_VENUE}
    assert (final.loc[final["venue"] == SPOT_VENUE, "taker_fee_bps"]
            == SPOT_TAKER_FEE_BPS).all()

    final.to_csv(FINAL_PATH, index=False)
    print(f"\nwrote {FINAL_PATH}  rows={len(final)}")
    print(f"provenance: {dict(final['half_spread_source'].value_counts())}")
    print(f"nan half_spread: {int(final['half_spread_bps'].isna().sum())}")
    for v in (FUTURES_VENUE, SPOT_VENUE):
        sub = final[final["venue"] == v]
        print(f"venue {v}: rows={len(sub)} "
              f"median_half_spread={sub['half_spread_bps'].median():.3f} "
              f"median_impact={sub['impact_bps_per_unit'].median():.4g}")
    return final


finalize()

## AR outcome

Ran Abdi-Ranaldo over the 46-week bookTicker overlap window on all 10 symbols. AR is not usable on this data.

- BTC, ETH, SOL, BNB, AVAX. The covariance term goes negative and the estimator clips to zero across most weeks.
- LINK, LTC, XRP, DOGE. Estimator produces roughly half the measured spread with no meaningful correlation to the book values (corr ~0).
- ADA. Only symbol where AR tracks book (corr 0.63, ratio 0.99). One out of ten is not enough to trust.

## Fallback: per-symbol book median

Filling the 22 months outside the bookTicker archive with a per-symbol constant equal to the median of that symbol's 46 measured book weeks. Loses time variation in the gap window but is grounded in real measurements and does not produce zero-cost fills. Provenance column marks each row as either `book` (measured for that week) or `book_median` (per-symbol constant fill). The AR helper stays in the notebook in case a future dataset (longer bookTicker window, wider-spread symbol universe) makes it viable.
